# 03 - Data Preparation

Step 3 data preparation pipeline for classification on `Revenue`.
Outputs are saved to `step3_prepare/tables/` as pickle files.


In [ ]:
import os
import pickle
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer

warnings.filterwarnings("ignore")

In [ ]:
# ---------- Global Paths ----------
# Reproducible root: run notebook from project root directory.
# If needed, replace os.getcwd() with your explicit coursework folder path.
PROJECT_ROOT = os.getcwd()
DATA_PATH = os.path.join(PROJECT_ROOT, "online_shoppers_intention.csv")
TARGET = "Revenue"

STEP_DIRS = {
    "step1": os.path.join(PROJECT_ROOT, "step1_obtain_frame"),
    "step2": os.path.join(PROJECT_ROOT, "step2_eda"),
    "step3": os.path.join(PROJECT_ROOT, "step3_prepare"),
    "step4": os.path.join(PROJECT_ROOT, "step4_shortlist"),
    "step5": os.path.join(PROJECT_ROOT, "step5_finetune"),
    "step6": os.path.join(PROJECT_ROOT, "step6_present"),
}

for step_dir in STEP_DIRS.values():
    os.makedirs(os.path.join(step_dir, "plots"), exist_ok=True)
    os.makedirs(os.path.join(step_dir, "tables"), exist_ok=True)

STEP3_DIR = STEP_DIRS["step3"]
STEP3_PLOT_DIR = os.path.join(STEP3_DIR, "plots")
STEP3_TABLE_DIR = os.path.join(STEP3_DIR, "tables")

print("Resolved PROJECT_ROOT:", PROJECT_ROOT)
print("Data path:", DATA_PATH)
print("Step3 tables:", STEP3_TABLE_DIR)


## Step 3.1 - Load Data and Basic Normalization


In [ ]:
def load_data(path: str) -> pd.DataFrame:
    """Load raw CSV dataset."""
    return pd.read_csv(path)


def normalize_binary_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize binary columns to 0/1 for consistent preprocessing.
    This keeps target and binary indicators model-ready.
    """
    out = df.copy()
    binary_map = {
        "true": 1, "false": 0,
        "yes": 1, "no": 0,
        "1": 1, "0": 0,
    }

    for col in ["Weekend", "Revenue"]:
        if col not in out.columns:
            continue

        if pd.api.types.is_bool_dtype(out[col]):
            out[col] = out[col].astype(int)
        elif out[col].dtype == object:
            mapped = out[col].astype(str).str.strip().str.lower().map(binary_map)
            if mapped.notna().all():
                out[col] = mapped.astype(int)

    return out

## Step 3.2 - Feature Groups and Preprocessing Design


In [ ]:
# Categorical features requested for one-hot encoding
CATEGORICAL_FEATURES = [
    "Month",
    "VisitorType",
    "OperatingSystems",
    "Browser",
    "TrafficType",
    "Region",
]

# Duration features requested for log1p transformation
LOG_DURATION_FEATURES = [
    "Administrative_Duration",
    "Informational_Duration",
    "ProductRelated_Duration",
]

# Feature to drop due to multicollinearity decision
DROP_FEATURES = ["BounceRates"]


def make_one_hot_encoder() -> OneHotEncoder:
    """
    Build an OneHotEncoder compatible with multiple sklearn versions.
    """
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_feature_lists(df: pd.DataFrame, target: str = TARGET):
    """
    Build final feature lists:
    - drop target + selected dropped features
    - apply requested categorical handling
    - split numeric into log-transformed and regular numeric groups
    """
    feature_df = df.drop(columns=[target], errors="ignore").copy()
    feature_df = feature_df.drop(columns=DROP_FEATURES, errors="ignore")

    categorical = [c for c in CATEGORICAL_FEATURES if c in feature_df.columns]

    numeric_all = feature_df.select_dtypes(include=[np.number, "bool"]).columns.tolist()
    log_numeric = [c for c in LOG_DURATION_FEATURES if c in numeric_all]
    regular_numeric = [c for c in numeric_all if c not in log_numeric and c not in categorical]

    return feature_df, categorical, log_numeric, regular_numeric


def build_preprocessor(categorical_features, log_numeric_features, regular_numeric_features) -> ColumnTransformer:
    """
    Build preprocessing pipeline with:
    1) Imputation (median numeric, mode categorical)
    2) log1p on selected duration features
    3) one-hot encoding on selected categorical features
    4) standard scaling on all numeric features
    """
    log_numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ("scaler", StandardScaler()),
    ])

    regular_numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_one_hot_encoder()),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("log_num", log_numeric_pipeline, log_numeric_features),
            ("num", regular_numeric_pipeline, regular_numeric_features),
            ("cat", categorical_pipeline, categorical_features),
        ],
        remainder="drop",
    )

    return preprocessor

## Step 3.3 - Train/Test Split and Transformation


In [ ]:
def extract_target_series(df: pd.DataFrame, target: str = TARGET) -> pd.Series:
    """Return target as integer series for stratified split and modeling."""
    y = df[target].copy()

    if pd.api.types.is_bool_dtype(y):
        y = y.astype(int)
    elif y.dtype == object:
        mapped = y.astype(str).str.strip().str.lower().map({"true": 1, "false": 0, "1": 1, "0": 0, "yes": 1, "no": 0})
        if mapped.notna().all():
            y = mapped.astype(int)
        else:
            y = y.astype("category").cat.codes

    return y.astype(int)


def to_feature_dataframe(preprocessor: ColumnTransformer, X_transformed, index) -> pd.DataFrame:
    """Convert transformed matrix to named DataFrame for auditability and export."""
    feature_names = preprocessor.get_feature_names_out()

    if hasattr(X_transformed, "toarray"):
        X_transformed = X_transformed.toarray()

    return pd.DataFrame(X_transformed, columns=feature_names, index=index)


def split_and_transform(df: pd.DataFrame, target: str = TARGET):
    """
    1) Build feature subsets
    2) Stratified 80/20 split (random_state=42)
    3) Fit preprocessor on train only
    4) Transform train/test
    """
    feature_df, categorical, log_numeric, regular_numeric = build_feature_lists(df, target=target)
    y = extract_target_series(df, target=target)

    X_train, X_test, y_train, y_test = train_test_split(
        feature_df,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y,
    )

    preprocessor = build_preprocessor(
        categorical_features=categorical,
        log_numeric_features=log_numeric,
        regular_numeric_features=regular_numeric,
    )

    X_train_transformed = preprocessor.fit_transform(X_train)
    X_test_transformed = preprocessor.transform(X_test)

    X_train_processed = to_feature_dataframe(preprocessor, X_train_transformed, X_train.index)
    X_test_processed = to_feature_dataframe(preprocessor, X_test_transformed, X_test.index)

    metadata = {
        "categorical_features": categorical,
        "log_numeric_features": log_numeric,
        "regular_numeric_features": regular_numeric,
        "dropped_features": DROP_FEATURES,
        "n_train": len(X_train_processed),
        "n_test": len(X_test_processed),
        "n_features_processed": X_train_processed.shape[1],
    }

    return X_train_processed, X_test_processed, y_train, y_test, preprocessor, metadata

## Step 3.4 - Save Processed Artifacts


In [ ]:
def save_pickle_object(obj, path: str) -> None:
    """Save object as pickle file."""
    with open(path, "wb") as f:
        pickle.dump(obj, f)


def save_prepared_outputs(X_train, X_test, y_train, y_test, metadata: dict) -> None:
    """Persist prepared datasets and metadata into Step 3 tables directory."""
    out_paths = {
        "X_train": os.path.join(STEP3_TABLE_DIR, "X_train_processed.pkl"),
        "X_test": os.path.join(STEP3_TABLE_DIR, "X_test_processed.pkl"),
        "y_train": os.path.join(STEP3_TABLE_DIR, "y_train.pkl"),
        "y_test": os.path.join(STEP3_TABLE_DIR, "y_test.pkl"),
        "metadata": os.path.join(STEP3_TABLE_DIR, "prep_metadata.pkl"),
    }

    save_pickle_object(X_train, out_paths["X_train"])
    save_pickle_object(X_test, out_paths["X_test"])
    save_pickle_object(y_train, out_paths["y_train"])
    save_pickle_object(y_test, out_paths["y_test"])
    save_pickle_object(metadata, out_paths["metadata"])

    print("Saved processed artifacts:")
    for k, v in out_paths.items():
        print(f"- {k}: {v}")

## Run Step 3 Pipeline


In [ ]:
# Load and normalize
df_raw = load_data(DATA_PATH)
df = normalize_binary_columns(df_raw)

print(f"Raw dataset shape: {df.shape}")
display(df.head())

# Split + preprocess
X_train_processed, X_test_processed, y_train, y_test, preprocessor, metadata = split_and_transform(df, target=TARGET)

print("\nPrepared dataset shapes:")
print("X_train_processed:", X_train_processed.shape)
print("X_test_processed:", X_test_processed.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

print("\nPipeline metadata:")
for k, v in metadata.items():
    print(f"{k}: {v}")

# Save artifacts as pickle files
save_prepared_outputs(X_train_processed, X_test_processed, y_train, y_test, metadata)

# Optional quick preview for report appendix
display(X_train_processed.head())
display(y_train.head())
